# MedCPT Article Encoder: ML Registry + SPCS Deployment

This notebook registers the **MedCPT Article Encoder** (`ncbi/MedCPT-Article-Encoder`) in the Snowflake ML Registry and deploys it as an SPCS service for bulk embedding generation.

**Why a separate Article Encoder?**
MedCPT is an asymmetric dual-encoder model from NCBI. Articles and queries live in the same 768-dim vector space but are encoded by different models. This notebook handles the article side - used to pre-compute embeddings for the entire PubMed corpus (~72M chunks).

**Flow:**
1. Download the HuggingFace model and save locally
2. Wrap it in a `CustomModel` class with batched `encode` inference API
3. Validate locally to confirm output shape (768-dim)
4. Register in Snowflake ML Registry (`MEDCPT_EMBEDDER`)
5. Deploy as an SPCS service (`MEDCPT_EMBEDDER_SVC`) on multi-GPU for bulk throughput
6. Test via Python SDK (`mv.run`) and SQL (`!encode`) against sample data

In [ ]:
!pip install sentence-transformers

## Setup: Session, Registry, and imports

Connect to Snowflake and initialize the ML Registry in `SFSE_PUBMED_DB.PUBMED`.

In [ ]:
from snowflake.ml.registry import Registry 
from snowflake.snowpark.context import get_active_session
import pandas as pd
import transformers



session = get_active_session()

session.sql("use schema sfse_pubmed_db.pubmed").collect()


reg = Registry(
    session=session,
    database_name="sfse_pubmed_db",
    schema_name="pubmed",
)



## Define the MedCPT Article Encoder custom model

Download `ncbi/MedCPT-Article-Encoder` from HuggingFace, save locally, and wrap it in a `CustomModel` subclass. The `encode` API processes input in batches of 256 for throughput - important since this model will embed ~72M chunks in the bulk historical run.

The local validation at the end confirms 768-dim embeddings are produced correctly before registering.

In [ ]:
from sentence_transformers import SentenceTransformer
from snowflake.ml.model import custom_model


BATCH_SIZE = 256
MODEL_NAME = "ncbi/MedCPT-Article-Encoder"
MODEL_LOCAL_DIR = "/tmp/medcpt_article_encoder"

local_model = SentenceTransformer(
    MODEL_NAME,
    trust_remote_code=True,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
local_model.save(MODEL_LOCAL_DIR)

class MedCPTArticleEmbedder(custom_model.CustomModel):
    def __init__(self, context: custom_model.ModelContext) -> None:
        super().__init__(context)
        self.model = SentenceTransformer(
            self.context["model_dir"],
            trust_remote_code=True,
            device="cuda" if torch.cuda.is_available() else "cpu",
        )
        self.batch_size = BATCH_SIZE

    @custom_model.inference_api
    def encode(self, input: pd.DataFrame) -> pd.DataFrame:
        texts = input["TEXT"].tolist()
        all_embeddings = []
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i : i + self.batch_size]
            embs = self.model.encode(batch, show_progress_bar=False, convert_to_numpy=True, batch_size=len(batch))
            all_embeddings.extend(embs.tolist())
        return pd.DataFrame({"EMBEDDING": all_embeddings})


mc = custom_model.ModelContext(
    artifacts={"model_dir": MODEL_LOCAL_DIR},
)

medcpt_model = MedCPTArticleEmbedder(mc)

sample_input = pd.DataFrame({
    "TEXT": ["This is a clinical sentence about cardiac health."] * 4
})

result = medcpt_model.encode(sample_input)
print(f"Output rows: {len(result)}")
print(f"Embedding dim: {len(result['EMBEDDING'].iloc[0])}")
print(f"First 5 values: {result['EMBEDDING'].iloc[0][:5]}")

## Register in ML Registry and deploy SPCS service

Log the model to the registry, then deploy as `MEDCPT_EMBEDDER_SVC` on `GPU_ML_M_POOL`. This configuration uses 4 GPUs per instance with 4 worker processes and 4 instances for maximum throughput during the bulk historical embed. `max_batch_rows=256` matches the model's internal batch size.

In [ ]:
mv = reg.log_model(
    medcpt_model,
    model_name="MEDCPT_EMBEDDER",
    version_name="v1",
    sample_input_data=sample_input,
    pip_requirements=["sentence-transformers", "torch", "transformers"],
    options={"use_gpu": True, "cuda_version": "12.3"},
)

mv.create_service(
    service_name="MEDCPT_EMBEDDER_SVC",
    service_compute_pool="GPU_ML_M_POOL",
    ingress_enabled=True,
    gpu_requests="4",
    min_instances=4,
    max_instances=4,
    num_workers=4,
    max_batch_rows=256,
)

## (Optional) Scale up - alternate service configuration

A second `create_service` call with a different GPU/instance configuration. Useful for experimenting with throughput vs. cost tradeoffs during the historical embed.

In [ ]:
mv.create_service(
    service_name="MEDCPT_EMBEDDER_SVC",
    service_compute_pool="GPU_ML_M_POOL",
    ingress_enabled=True,
    gpu_requests="4",
    num_workers=4,
    min_instances=6
    max_instances=6,
    max_batch_rows=128
)

## Test: Embed via Python SDK (`mv.run`)

Call the deployed SPCS service through the ML Registry Python API. Pulls 10 rows from `pubmed_oa_test_set`, passes the CHUNK text to the encoder, and returns embeddings.

In [ ]:
from snowflake.snowpark.functions import parse_json, col
from snowflake.snowpark.types import VectorType


df = session.table('pubmed_oa_test_set').limit(10)

input_df = pd.DataFrame({"TEXT": df["CHUNK"].tolist()})
result = mv.run(input_df, function_name="encode", service_name="medcpt_embedder_svc")

result

## Test: Embed via SQL (`!encode`)

Call the SPCS service directly from SQL using the `service!function` syntax. This is the same pattern used in the bulk historical embed (script 2) and the daily incremental load procedure (script 6). The `:EMBEDDING::VECTOR(FLOAT, 768)` extracts and casts the embedding from the service's JSON response.

In [ ]:
sql = """
select *, MEDCPT_EMBEDDER_SVC!encode(CHUNK):EMBEDDING::VECTOR(FLOAT, 768) AS EMBEDDING
from pubmed_oa_test_set"""

df = session.sql(sql)
df